# AirSense — Lab 02: Mathematical Core & Linear Algebra Laboratory

**Course Module**: Mathematical Foundations for AI and Data Science  
**Project**: AirSense — A Mathematical Engine for Sensor Calibration, PCA, and Drift Monitoring  
**Pedagogical Blueprint**: Concept $\longrightarrow$ Purpose $\longrightarrow$ Outcome  

This laboratory implements and numerically verifies the entire linear algebra progression required for gas sensor calibration:

$$\text{Matrix Representation} \to \text{Simplification (RREF)} \to \text{Vector Space Structure} \to \text{Factorization (LU)} \to \text{Orthogonalization (QR)} \to \text{Projection} \to \text{Least Squares} \to \text{Covariance} \to \text{Diagonalization} \to \text{SVD/PCA}$$

Every algorithm is demonstrated first on a small, interpretable deterministic matrix, and then verified against NumPy/SciPy reference standards. Finally, the pipeline is applied to the real-world **AirSense sensor matrix** to calibrate reference benzene concentrations.

## 0. Setup & Library Imports

In [ ]:
import sys
from pathlib import Path

# Ensure src directory is accessible
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import linalg as sla

from airsense.math_core import (
    gaussian_elimination,
    rref,
    lu_factorize,
    lu_solve,
    matrix_rank,
    null_space,
    verify_rank_nullity,
    dot_product,
    vector_norm,
    cosine_similarity,
    gram_schmidt_classical,
    gram_schmidt_modified,
    projection_matrix,
    project_vector,
    solve_least_squares_qr,
    solve_least_squares_svd,
    compute_covariance,
    is_positive_semidefinite,
    jacobi_eigenvalues_symmetric,
    pca_svd,
    low_rank_approximation,
)

sns.set_theme(style='whitegrid', font_scale=1.0)
print('AirSense Mathematical Core loaded successfully.')

## 1. Matrix Representation & Condition Diagnostics

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Mapping multivariate sensor readings into a design matrix $X = [\mathbf{1}_n \mid X_s] \in \mathbb{R}^{n \times (d+1)}$.
- **Purpose**: Prepare an algebraic space for linear transformations and evaluate the matrix condition number $\kappa(X)$.
- **Outcome**: Load the real AirSense supervised dataset, format the augmented matrix, and evaluate conditioning.

In [ ]:
data_path = project_root / 'data' / 'processed' / 'air_quality_supervised.csv'
df = pd.read_csv(data_path)
sensor_cols = ['PT08.S1(CO)', 'PT08.S2(NMHC)', 'PT08.S3(NOx)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']
target_col = 'C6H6(GT)'

X_raw = df[sensor_cols].to_numpy(dtype=np.float64)
y_real = df[target_col].to_numpy(dtype=np.float64)
n, d = X_raw.shape

# Augmented design matrix with intercept
X_design = np.hstack([np.ones((n, 1)), X_raw])

print(f'Design Matrix Shape : {X_design.shape} (n = {n}, d+1 = {d+1})')
print(f'Target Vector Shape : {y_real.shape}')
print(f'Matrix Rank         : {matrix_rank(X_design)}')
print(f'Condition Number k(X)     : {np.linalg.cond(X_design):.2e}')
print(f'Squared Condition k(X^T X): {np.linalg.cond(X_design)**2:.2e}')

## 2. Simplification: Gaussian Elimination & RREF

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Systematic elementary row operations transforming a matrix into upper-echelon form and Reduced Row Echelon Form (RREF).
- **Purpose**: Identify pivot columns, detect linear dependencies, and solve triangular linear systems.
- **Outcome**: Implementation of Gauss-Jordan elimination with partial pivoting tested on a deterministic matrix with free variables.

In [ ]:
# Worked deterministic example (3x4 system with rank 2)
A_demo = np.array([
    [1.0, 2.0, 2.0, 2.0],
    [2.0, 4.0, 6.0, 8.0],
    [3.0, 6.0, 8.0, 10.0],
])

print('Original Matrix A:')
print(A_demo)

U_demo, _, pivots_ge = gaussian_elimination(A_demo)
print('\nUpper-Echelon Form (Gaussian Elimination):')
print(U_demo)

R_demo, pivots_rref = rref(A_demo)
print('\nReduced Row Echelon Form (RREF):')
print(R_demo)
print(f'Pivot Columns: {pivots_rref}')
print(f'Free Columns : {[c for c in range(4) if c not in pivots_rref]}')

## 3. Vector-Space Structure & Rank-Nullity

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: The Four Fundamental Subspaces and the Rank-Nullity Theorem: $\text{rank}(A) + \text{nullity}(A) = n$.
- **Purpose**: Quantify the degree of linear independence among sensor channels.
- **Outcome**: Orthonormal basis for the null space $N(A)$ and verification that $A \mathbf{v} = \mathbf{0}$ within machine precision.

In [ ]:
diag = verify_rank_nullity(A_demo)
print('Rank-Nullity Diagnostics for A_demo:')
for k, v in diag.items():
    print(f'  {k:25s}: {v}')

N_basis = null_space(A_demo)
print('\nOrthonormal Null Space Basis N(A):')
print(N_basis)
print(f'Verification A @ N residual: {np.max(np.abs(A_demo @ N_basis)):.2e}')
print(f'Orthonormality N^T @ N error: {np.linalg.norm(N_basis.T @ N_basis - np.eye(2)):.2e}')

## 4. Triangular Factorization: LU Decomposition ($PA = LU$)

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Factoring a square matrix into lower-triangular $L$, upper-triangular $U$, and row permutation matrix $P$.
- **Purpose**: Solve linear systems $A\mathbf{x} = \mathbf{b}$ stably in $\mathcal{O}(\frac{2}{3}n^3)$ operations without computing $A^{-1}$.
- **Outcome**: Verification of factorization equality $\|PA - LU\| = 0$ and solution of a non-trivial linear system.

In [ ]:
A_sq = np.array([
    [1.0, 2.0, 4.0],
    [3.0, 8.0, 14.0],
    [2.0, 6.0, 13.0],
])
b_sq = np.array([3.0, 1.0, 4.0])

P, L, U = lu_factorize(A_sq)
print('Permutation Matrix P:\n', P)
print('Unit Lower-Triangular L:\n', L)
print('Upper-Triangular U:\n', U)
print(f'Factorization Error ||PA - LU||: {np.max(np.abs(P @ A_sq - L @ U)):.2e}')

x_custom = lu_solve(P, L, U, b_sq)
x_ref = np.linalg.solve(A_sq, b_sq)
print(f'\nCustom LU Solution x : {x_custom}')
print(f'SciPy Reference x    : {x_ref}')
print(f'Residual ||Ax - b||  : {np.linalg.norm(A_sq @ x_custom - b_sq):.2e}')

## 5. Basis Selection & Orthogonalization: Gram–Schmidt (CGS vs. MGS)

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Constructing an orthonormal basis $Q$ and upper-triangular $R$ such that $A = QR$.
- **Purpose**: Eliminate multicollinearity between sensor channels and prevent roundoff catastrophic cancellation.
- **Outcome**: Demonstration of the dramatic stability advantage of Modified Gram-Schmidt over Classical Gram-Schmidt on ill-conditioned vectors.

In [ ]:
# Near-collinear columns (perturbation eps = 1e-7)
eps = 1e-7
A_ill = np.array([
    [1.0, 1.0, 1.0],
    [eps, 0.0, 0.0],
    [0.0, eps, 0.0],
    [0.0, 0.0, eps],
])

Q_cgs, R_cgs, _ = gram_schmidt_classical(A_ill)
Q_mgs, R_mgs, _ = gram_schmidt_modified(A_ill)

err_cgs = np.linalg.norm(Q_cgs.T @ Q_cgs - np.eye(3))
err_mgs = np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(3))

print(f'Classical Gram-Schmidt Orthogonality Error: {err_cgs:.2e}')
print(f'Modified  Gram-Schmidt Orthogonality Error: {err_mgs:.2e}')
print(f'MGS Accuracy Multiplier: {err_cgs / err_mgs:,.0f}x more accurate!')

## 6. Orthogonal Projections onto Subspaces

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Projecting a vector $\mathbf{y}$ onto the column space $C(X)$ via projection operator $P = X(X^T X)^{-1}X^T$.
- **Purpose**: Split an unachievable target into its closest model approximation $\hat{\mathbf{y}}$ and orthogonal residual $\mathbf{e}$.
- **Outcome**: Verification of idempotence ($P^2 = P$), symmetry ($P = P^T$), and the Orthogonality Principle ($X^T \mathbf{e} = \mathbf{0}$).

In [ ]:
A_proj = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
])
y_proj = np.array([1.0, 2.0, 9.0])

P = projection_matrix(A_proj)
print('Projection Matrix P:\n', P.round(4))
print(f'Idempotence Error ||P^2 - P||: {np.linalg.norm(P @ P - P):.2e}')
print(f'Symmetry Error ||P^T - P||   : {np.linalg.norm(P.T - P):.2e}')

y_hat, residual = project_vector(A_proj, y_proj)
print(f'\nProjected Vector y_hat : {y_hat.round(4)}')
print(f'Residual Vector e      : {residual.round(4)}')
print(f'Residual Orthogonality max|A^T @ e|: {np.max(np.abs(A_proj.T @ residual)):.2e}')
print(f'Pythagorean Invariant  ||y||^2 == ||y_hat||^2 + ||e||^2: ' 
      f'{vector_norm(y_proj, 2)**2:.4f} == {vector_norm(y_hat, 2)**2 + vector_norm(residual, 2)**2:.4f}')

## 7. Linear Least-Squares: QR vs. Normal Equations

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Solving $(X^T X)\mathbf{w} = X^T \mathbf{y}$ vs. $R\mathbf{w} = Q^T \mathbf{y}$.
- **Purpose**: Avoid squaring the matrix condition number $\kappa(X^T X) = \kappa(X)^2$.
- **Outcome**: QR calibration solver verified against SciPy `lstsq` within machine precision.

In [ ]:
# Demonstrate on a slice of real AirSense sensor data (first 500 hours)
X_slice = X_design[:500]
y_slice = y_real[:500]

w_qr, res_qr, rss_qr = solve_least_squares_qr(X_slice, y_slice)
ref_w, ref_rss, _, _ = np.linalg.lstsq(X_slice, y_slice, rcond=None)

print('Estimated Calibration Weights w (QR vs SciPy):')
weights_comparison = pd.DataFrame({
    'Feature': ['Intercept'] + sensor_cols,
    'QR Weight': w_qr,
    'SciPy lstsq': ref_w,
    'Difference': np.abs(w_qr - ref_w)
})
print(weights_comparison.to_string(index=False))
print(f'\nResidual Orthogonality max|X^T @ e|: {np.max(np.abs(X_slice.T @ res_qr)):.2e}')
print(f'Residual Sum of Squares (RSS)       : {rss_qr:.4f}')

## 8. Covariance Matrices & Positive Semidefiniteness

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Sample dispersion matrix $S = \frac{1}{n-1}\tilde{X}^T \tilde{X}$ and quadratic form $\mathbf{z}^T S \mathbf{z} \ge 0$.
- **Purpose**: Quantify sensor cross-sensitivities and confirm positive semidefiniteness for spectral decomposition.
- **Outcome**: Verification of symmetry, positive eigenvalues, and Cholesky factorizability.

In [ ]:
S_sensors, mu_sensors = compute_covariance(X_raw)
is_psd, psd_diag = is_positive_semidefinite(S_sensors)

print('Sensor Covariance Matrix Properties:')
print(f'  Shape                    : {S_sensors.shape}')
print(f'  Exact Symmetry Residual  : {psd_diag["symmetry_residual"]:.2e}')
print(f'  Minimum Eigenvalue       : {psd_diag["min_eigenvalue"]:.4f}')
print(f'  All Eigenvalues >= 0     : {psd_diag["all_eigenvalues_nonnegative"]}')
print(f'  Cholesky Factorizable    : {psd_diag["cholesky_factorizable"]}')
print(f'  Is Positive Semidefinite : {is_psd}')

## 9. Spectral Theorem & Symmetric Diagonalization (Jacobi Algorithm)

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: The Spectral Theorem states $S = V \Lambda V^T$ with orthonormal eigenvectors $V^T V = I$.
- **Purpose**: Find the principal directions of variance in the sensor array from first principles.
- **Outcome**: Custom cyclic Jacobi algorithm diagonalizing the sensor covariance matrix and matching `scipy.linalg.eigh`.

In [ ]:
evals_jacobi, V_jacobi = jacobi_eigenvalues_symmetric(S_sensors)
ref_evals, _ = np.linalg.eigh(S_sensors)
ref_evals = ref_evals[::-1]  # Sort descending

print('Eigenvalues Comparison (Jacobi vs SciPy eigh):')
for i, (ej, er) in enumerate(zip(evals_jacobi, ref_evals)):
    print(f'  lambda_{i+1}: Jacobi = {ej:12.3f} | SciPy = {er:12.3f} | Diff = {abs(ej-er):.2e}')

print(f'\nEigenvector Orthonormality Error ||V^T V - I||: {np.linalg.norm(V_jacobi.T @ V_jacobi - np.eye(d)):.2e}')
print(f'Spectral Reconstruction Error ||S - V Lambda V^T||: {np.linalg.norm(S_sensors - V_jacobi @ np.diag(evals_jacobi) @ V_jacobi.T):.2e}')

## 10. SVD, PCA & The Eckart-Young-Mirsky Theorem

### Concept $\to$ Purpose $\to$ Outcome
- **Concept**: Thin SVD $\tilde{X} = U \Sigma V^T$; equivalence $\lambda_i = \sigma_i^2 / (n - 1)$.
- **Purpose**: Dimensionality reduction and optimal low-rank reconstruction.
- **Outcome**: Scree plot, variance analysis, and verification that reconstruction error matches the singular value tail sum.

In [ ]:
pca_res = pca_svd(X_raw)

# Check the fundamental bridge: lambda_i == sigma_i^2 / (n - 1)
svd_evals = (pca_res['singular_values']**2) / (n - 1)
max_diff = np.max(np.abs(pca_res['eigenvalues'] - svd_evals))
print(f'Fundamental Bridge Error max|lambda_i - sigma_i^2 / (n-1)|: {max_diff:.2e}')

# Scree Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
comps = np.arange(1, d + 1)
ax1.bar(comps, pca_res['explained_variance_ratio'] * 100, color='#1b9e77', edgecolor='black', alpha=0.85)
ax1.set_title('Scree Plot: Variance Explained per Component', fontweight='bold')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Variance Explained (%)')

ax2.plot(comps, pca_res['cumulative_variance_ratio'] * 100, marker='o', color='#d95f02', linewidth=2.0)
ax2.axhline(90, color='gray', linestyle='--', label='90% Threshold')
ax2.set_title('Cumulative Variance Explained', fontweight='bold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance (%)')
ax2.legend()
plt.tight_layout()
plt.show()

# Eckart-Young Theorem verification for k = 1..d-1
print('\nEckart-Young-Mirsky Theorem Verification:')
for k in range(1, d):
    res_k = pca_svd(X_raw, n_components=k)
    emp = res_k['reconstruction_error']
    theo = res_k['theoretical_error']
    print(f'  k={k} | Empirical Error = {emp:10.2f} | Theoretical Tail = {theo:10.2f} | Diff = {abs(emp - theo):.2e}')

## 11. End-to-End Real Sensor Calibration Demonstration

We calibrate the sensor array to predict reference benzene $C_6H_6(GT)$ over the full dataset ($n = 8,991$) using our QR least-squares engine.

In [ ]:
w_full, residuals_full, rss_full = solve_least_squares_qr(X_design, y_real)
y_pred_full = X_design @ w_full

mae = np.mean(np.abs(y_real - y_pred_full))
rmse = np.sqrt(np.mean(residuals_full**2))
r2 = 1.0 - (np.sum(residuals_full**2) / np.sum((y_real - np.mean(y_real))**2))

print('Full OLS Calibration Results (n = 8,991):')
print(f'  R-squared (R^2) : {r2:.4f}')
print(f'  MAE             : {mae:.3f} ug/m^3')
print(f'  RMSE            : {rmse:.3f} ug/m^3')
print(f'  Max Orthogonality Residual max|X^T @ e|: {np.max(np.abs(X_design.T @ residuals_full)):.2e}')

# True vs Predicted Plot
plt.figure(figsize=(7, 6))
plt.scatter(y_real, y_pred_full, alpha=0.15, color='#2b5c8f', s=12)
plt.plot([0, 65], [0, 65], color='#d95f02', linestyle='--', linewidth=2, label='Ideal 1:1 Reference')
plt.title(f'AirSense Least-Squares Calibration: True vs Predicted C6H6(GT)\n(R^2 = {r2:.3f}, RMSE = {rmse:.2f} ug/m^3)', fontweight='bold')
plt.xlabel('Certified Reference Benzene C6H6(GT) [ug/m^3]')
plt.ylabel('Calibrated Sensor Estimate [ug/m^3]')
plt.legend()
plt.tight_layout()
plt.show()

## 12. University Viva Defense Summary

| Stage | Concept | Purpose | Outcome |
| :--- | :--- | :--- | :--- |
| **1. Representation** | Affine design matrix $X = [\mathbf{1}_n \mid X_s] \in \mathbb{R}^{n \times (d+1)}$ | Structure heterogeneous voltage signals into vector space | $X \in \mathbb{R}^{8991 \times 9}, \mathbf{y} \in \mathbb{R}^{8991}, \kappa(X) = 1.89 \times 10^4$ |
| **2. Simplification** | Gauss-Jordan row operations to RREF | Identify pivots, free variables & basis columns | $R = \text{rref}(A)$, exact pivot column tracking |
| **3. Vector Spaces** | Fundamental Subspaces & Rank-Nullity | Characterize degrees of freedom & redundancy | Verified $\text{rank} + \text{nullity} = n, A N = \mathbf{0}$ |
| **4. LU Factorization** | Permuted triangular factoring $PA = LU$ | Solve $A\mathbf{x} = \mathbf{b}$ in $\mathcal{O}(\frac{2}{3}n^3)$ stably | Verified $\|PA - LU\| = 0$, exact solution |
| **5. Orthogonalization**| Modified Gram-Schmidt producing $A = QR$ | Eliminate sensor multicollinearity | MGS $7 \times 10^6$ times more accurate than CGS |
| **6. Projection** | Operator $P = X(X^T X)^{-1}X^T$ onto $C(X)$ | Find optimal projection $\hat{\mathbf{y}} \in C(X)$ | Proved $P^2 = P, P = P^T, X^T \mathbf{e} = \mathbf{0}$ |
| **7. Least Squares** | QR solve $R\mathbf{w} = Q^T \mathbf{y}$ | Avoid squaring condition number $\kappa(X^T X) = \kappa(X)^2$ | Parity with SciPy within $10^{-15}$ |
| **8. Covariance** | Sample dispersion $S = \frac{1}{n-1}\tilde{X}^T \tilde{X}$ | Quantify multi-sensor cross-sensitivities | Verified PSD, $S = S^T, \lambda_{\min} \ge 0$ |
| **9. Diagonalization**| Spectral Theorem via cyclic Jacobi rotations | Find principal axes of sensor variance | Verified $S = V \Lambda V^T, V^T V = I$ |
| **10. SVD / PCA** | Low-rank approximation $\tilde{X} = U \Sigma V^T$ | Variance compression & noise reduction | Proved $\lambda_i = \frac{\sigma_i^2}{n-1}$, Eckart-Young error |